# Brain Tumor Segmentation using U-Net with Transfer Learning
## BraTS 2021 Dataset Analysis and Model Development

## Import Required Libraries

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import nibabel as nib
from scipy import ndimage
import warnings
warnings.filterwarnings('ignore')

print("TensorFlow version:", tf.__version__)
print("Available GPUs:", len(tf.config.list_physical_devices('GPU')))

## Load and Explore BraTS Dataset

Note: Download BraTS 2021 dataset from [Kaggle](https://www.kaggle.com/datasets/dschettler8845/brats-2021-task1). The structure should have train_images and train_masks folders.

In [ ]:
DATASET_PATH = '/path/to/BraTS2021/'
TRAIN_IMAGES_PATH = os.path.join(DATASET_PATH, 'train_images')
TRAIN_MASKS_PATH = os.path.join(DATASET_PATH, 'train_masks')

if not os.path.exists(DATASET_PATH):
    raise FileNotFoundError(
        f"Dataset not found at {DATASET_PATH}. "
        f"Please download the BraTS 2021 dataset from "
        f"https://www.kaggle.com/datasets/dschettler8845/brats-2021-task1 "
        f"and set DATASET_PATH to the correct location."
    )

print(f"Dataset found at {DATASET_PATH}")
print("Train images:", len(os.listdir(TRAIN_IMAGES_PATH)) if os.path.exists(TRAIN_IMAGES_PATH) else 0)
print("Train masks:", len(os.listdir(TRAIN_MASKS_PATH)) if os.path.exists(TRAIN_MASKS_PATH) else 0)

## Preprocess MRI Data

In [ ]:
def load_nifti_file(filepath):
    scan = nib.load(filepath)
    scan_data = scan.get_fdata()
    return scan_data

def preprocess_scan(scan):
    brain_mask = scan > np.mean(scan)
    scan = scan * brain_mask
    scan = (scan - np.mean(scan)) / (np.std(scan) + 1e-8)
    return scan

def load_mri_volume(image_path, modalities=['T1', 'T1c', 'T2', 'FLAIR']):
    volume_data = {}
    for modality in modalities:
        file_path = image_path.replace('T1', modality)
        if os.path.exists(file_path):
            try:
                volume_data[modality] = load_nifti_file(file_path)
            except:
                print(f"Could not load {file_path}")
    return volume_data

def load_segmentation_mask(mask_path):
    try:
        mask = load_nifti_file(mask_path)
        return mask
    except:
        return None

print("Preprocessing functions defined successfully")

## Convert 3D Volumes to 2D Slices

In [ ]:
def convert_3d_to_2d_slices(volume, mask, min_tumor_pixels=100):
    slices_list = []
    masks_list = []
    
    for i in range(volume.shape[2]):
        slice_2d = volume[:, :, i]
        mask_2d = mask[:, :, i]
        
        if np.sum(mask_2d) > min_tumor_pixels:
            slices_list.append(slice_2d)
            masks_list.append(mask_2d)
    
    return slices_list, masks_list

def extract_all_slices_from_dataset(images_list, masks_list):
    all_images = []
    all_masks = []
    
    for idx, (img, mask) in enumerate(zip(images_list, masks_list)):
        if img.ndim == 3 and mask.ndim == 3:
            img_slices, mask_slices = convert_3d_to_2d_slices(img, mask)
            all_images.extend(img_slices)
            all_masks.extend(mask_slices)
    
    return np.array(all_images), np.array(all_masks)

print("3D to 2D conversion functions defined successfully")

## Normalize and Resize Images

In [ ]:
from tensorflow.keras.preprocessing.image import smart_resize

def normalize_image(image):
    image = (image - np.mean(image)) / (np.std(image) + 1e-8)
    return image

def resize_image(image, target_size=(256, 256)):
    image = np.expand_dims(image, axis=-1)
    image = smart_resize(image, target_size)
    return image.squeeze()

def preprocess_dataset(images, masks, target_size=(256, 256)):
    processed_images = []
    processed_masks = []
    
    for img, mask in zip(images, masks):
        normalized_img = normalize_image(img)
        resized_img = resize_image(normalized_img, target_size)
        resized_mask = resize_image(mask, target_size)
        
        processed_images.append(resized_img)
        processed_masks.append(resized_mask)
    
    return np.array(processed_images), np.array(processed_masks)

print("Normalization and resizing functions defined successfully")

## Prepare Input-Mask Datasets

In [ ]:
images_list = []
masks_list = []

for file in sorted(os.listdir(TRAIN_IMAGES_PATH)):
    if file.endswith('.nii') or file.endswith('.nii.gz'):
        img_path = os.path.join(TRAIN_IMAGES_PATH, file)
        mask_path = os.path.join(TRAIN_MASKS_PATH, file)
        
        if os.path.exists(mask_path):
            try:
                img_volume = load_nifti_file(img_path)
                mask_volume = load_segmentation_mask(mask_path)
                
                if mask_volume is not None:
                    images_list.append(img_volume)
                    masks_list.append(mask_volume)
            except Exception as e:
                print(f"Error loading {file}: {str(e)}")
                continue

if len(images_list) == 0:
    raise ValueError(
        "No valid MRI volumes and segmentation masks found in the dataset. "
        "Please verify that the dataset is properly structured with image and mask files."
    )

print(f"Loaded {len(images_list)} MRI volumes")

all_images, all_masks = extract_all_slices_from_dataset(images_list, masks_list)
print(f"Extracted {len(all_images)} 2D slices from 3D volumes")

X_processed, y_processed = preprocess_dataset(all_images, all_masks)
print(f"Processed dataset: Images shape {X_processed.shape}, Masks shape {y_processed.shape}")

## Build U-Net Model with Transfer Learning

In [ ]:
def dice_coefficient(y_true, y_pred, smooth=1e-6):
    y_true_flat = tf.reshape(y_true, [-1])
    y_pred_flat = tf.reshape(y_pred, [-1])
    intersection = tf.reduce_sum(y_true_flat * y_pred_flat)
    union = tf.reduce_sum(y_true_flat) + tf.reduce_sum(y_pred_flat)
    dice = (2 * intersection + smooth) / (union + smooth)
    return dice

def dice_loss(y_true, y_pred):
    return 1 - dice_coefficient(y_true, y_pred)

def iou(y_true, y_pred, smooth=1e-6):
    intersection = tf.reduce_sum(y_true * y_pred)
    union = tf.reduce_sum(y_true) + tf.reduce_sum(y_pred) - intersection
    iou_score = (intersection + smooth) / (union + smooth)
    return iou_score

def build_unet_with_transfer_learning(input_shape=(256, 256, 1)):
    inputs = layers.Input(shape=input_shape)
    
    x = layers.Conv2D(3, 1)(inputs)
    
    base_model = ResNet50(
        input_shape=(256, 256, 3),
        include_top=False,
        weights='imagenet'
    )
    
    for layer in base_model.layers:
        layer.trainable = False
    
    x = base_model(x)
    
    x = layers.UpSampling2D((2, 2))(x)
    x = layers.Conv2D(512, 3, padding='same', activation='relu')(x)
    x = layers.BatchNormalization()(x)
    
    x = layers.UpSampling2D((2, 2))(x)
    x = layers.Conv2D(256, 3, padding='same', activation='relu')(x)
    x = layers.BatchNormalization()(x)
    
    x = layers.UpSampling2D((2, 2))(x)
    x = layers.Conv2D(128, 3, padding='same', activation='relu')(x)
    x = layers.BatchNormalization()(x)
    
    x = layers.UpSampling2D((2, 2))(x)
    x = layers.Conv2D(64, 3, padding='same', activation='relu')(x)
    x = layers.BatchNormalization()(x)
    
    x = layers.Conv2D(32, 3, padding='same', activation='relu')(x)
    outputs = layers.Conv2D(1, 1, padding='same', activation='sigmoid')(x)
    
    model = models.Model(inputs=inputs, outputs=outputs)
    return model

model = build_unet_with_transfer_learning()
print(model.summary())

## Compile and Train the Model

In [ ]:
model.compile(
    optimizer=Adam(learning_rate=1e-4),
    loss=dice_loss,
    metrics=[dice_coefficient, iou]
)

data_augmentation = ImageDataGenerator(
    rotation_range=20,
    width_shift_range=0.1,
    height_shift_range=0.1,
    horizontal_flip=True,
    zoom_range=0.2,
    fill_mode='nearest'
)

early_stopping = EarlyStopping(
    monitor='val_loss',
    patience=10,
    restore_best_weights=True,
    verbose=1
)

model_checkpoint = ModelCheckpoint(
    'best_unet_model.h5',
    monitor='val_dice_coefficient',
    save_best_only=True,
    mode='max',
    verbose=1
)

history = model.fit(
    data_augmentation.flow(X_train, y_train, batch_size=16),
    steps_per_epoch=len(X_train) // 16,
    epochs=30,
    validation_data=(X_val, y_val),
    callbacks=[early_stopping, model_checkpoint],
    verbose=1
)

print("Training completed!")

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

axes[0, 0].plot(history.history['loss'], label='Training Loss')
axes[0, 0].plot(history.history['val_loss'], label='Validation Loss')
axes[0, 0].set_title('Model Loss')
axes[0, 0].set_xlabel('Epoch')
axes[0, 0].set_ylabel('Loss')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

axes[0, 1].plot(history.history['dice_coefficient'], label='Training Dice')
axes[0, 1].plot(history.history['val_dice_coefficient'], label='Validation Dice')
axes[0, 1].set_title('Dice Coefficient')
axes[0, 1].set_xlabel('Epoch')
axes[0, 1].set_ylabel('Dice Score')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

axes[1, 0].plot(history.history['iou'], label='Training IoU')
axes[1, 0].plot(history.history['val_iou'], label='Validation IoU')
axes[1, 0].set_title('Intersection over Union (IoU)')
axes[1, 0].set_xlabel('Epoch')
axes[1, 0].set_ylabel('IoU Score')
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

axes[1, 1].axis('off')
axes[1, 1].text(0.1, 0.5, 'Training Summary:\n' + 
                f'Final Training Loss: {history.history["loss"][-1]:.4f}\n' +
                f'Final Validation Loss: {history.history["val_loss"][-1]:.4f}\n' +
                f'Best Dice Score: {max(history.history["val_dice_coefficient"]):.4f}\n' +
                f'Best IoU Score: {max(history.history["val_iou"]):.4f}',
                fontsize=12, verticalalignment='center',
                bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

plt.tight_layout()
plt.show()

## Evaluate Model Performance

In [ ]:
y_pred = model.predict(X_test)

test_loss, test_dice, test_iou = model.evaluate(X_test, y_test, verbose=0)

print("=" * 50)
print("TEST SET EVALUATION METRICS")
print("=" * 50)
print(f"Test Loss: {test_loss:.4f}")
print(f"Test Dice Coefficient: {test_dice:.4f}")
print(f"Test IoU Score: {test_iou:.4f}")
print("=" * 50)

y_pred_binary = (y_pred > 0.5).astype(np.float32)
y_test_flat = y_test.reshape(-1)
y_pred_flat = y_pred_binary.reshape(-1)

dice_scores = []
iou_scores = []

for i in range(len(y_test)):
    intersection = np.sum(y_test[i] * y_pred_binary[i])
    union = np.sum(y_test[i]) + np.sum(y_pred_binary[i]) - intersection
    
    dice = 2 * intersection / (np.sum(y_test[i]) + np.sum(y_pred_binary[i]) + 1e-6)
    iou_score = intersection / (union + 1e-6)
    
    dice_scores.append(dice)
    iou_scores.append(iou_score)

print(f"Mean Dice Score: {np.mean(dice_scores):.4f} ± {np.std(dice_scores):.4f}")
print(f"Mean IoU Score: {np.mean(iou_scores):.4f} ± {np.std(iou_scores):.4f}")

per_class_metrics = {
    'Background Pixels': {
        'TP': np.sum((y_test_flat == 0) & (y_pred_flat == 0)),
        'TN': np.sum((y_test_flat == 0) & (y_pred_flat == 0)),
        'FP': np.sum((y_test_flat == 0) & (y_pred_flat == 1)),
        'FN': np.sum((y_test_flat == 0) & (y_pred_flat == 1)),
    },
    'Tumor Pixels': {
        'TP': np.sum((y_test_flat == 1) & (y_pred_flat == 1)),
        'TN': np.sum((y_test_flat == 1) & (y_pred_flat == 1)),
        'FP': np.sum((y_test_flat == 1) & (y_pred_flat == 0)),
        'FN': np.sum((y_test_flat == 1) & (y_pred_flat == 0)),
    }
}

print("\nPer-Class Metrics:")
for class_name, metrics in per_class_metrics.items():
    sensitivity = metrics['TP'] / (metrics['TP'] + metrics['FN'] + 1e-6)
    specificity = metrics['TN'] / (metrics['TN'] + metrics['FP'] + 1e-6)
    print(f"\n{class_name}:")
    print(f"  Sensitivity (Recall): {sensitivity:.4f}")
    print(f"  Specificity: {specificity:.4f}")

## Visualize Predictions vs Ground Truth

In [ ]:
def create_mask_overlay(image, mask, alpha=0.5):
    image_rgb = np.stack([image] * 3, axis=-1)
    mask_rgb = np.zeros_like(image_rgb)
    mask_rgb[..., 0] = mask
    
    overlay = (1 - alpha) * image_rgb + alpha * mask_rgb
    return overlay

num_samples = 9
fig, axes = plt.subplots(num_samples, 4, figsize=(16, 20))

for idx in range(num_samples):
    test_idx = idx
    
    image = X_test[test_idx].squeeze()
    ground_truth = y_test[test_idx].squeeze()
    prediction = y_pred[test_idx].squeeze()
    prediction_binary = (prediction > 0.5).astype(np.float32)
    
    axes[idx, 0].imshow(image, cmap='gray')
    axes[idx, 0].set_title(f'MRI Slice {test_idx+1}')
    axes[idx, 0].axis('off')
    
    axes[idx, 1].imshow(ground_truth, cmap='gray')
    axes[idx, 1].set_title(f'Ground Truth Mask {test_idx+1}')
    axes[idx, 1].axis('off')
    
    axes[idx, 2].imshow(prediction, cmap='gray')
    axes[idx, 2].set_title(f'Predicted Mask {test_idx+1}')
    axes[idx, 2].axis('off')
    
    overlay = create_mask_overlay(image, prediction_binary)
    axes[idx, 3].imshow(overlay)
    axes[idx, 3].set_title(f'Overlay {test_idx+1}')
    axes[idx, 3].axis('off')

plt.tight_layout()
plt.show()

print("Visualization completed!")

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 12))

y_pred_binary = (y_pred > 0.5).astype(np.int32).reshape(-1)
y_test_binary = (y_test > 0.5).astype(np.int32).reshape(-1)

cm = confusion_matrix(y_test_binary, y_pred_binary)

sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0, 0])
axes[0, 0].set_title('Confusion Matrix')
axes[0, 0].set_ylabel('Ground Truth')
axes[0, 0].set_xlabel('Prediction')

axes[0, 1].hist(dice_scores, bins=20, edgecolor='black', alpha=0.7)
axes[0, 1].set_title('Dice Score Distribution')
axes[0, 1].set_xlabel('Dice Score')
axes[0, 1].set_ylabel('Frequency')
axes[0, 1].axvline(np.mean(dice_scores), color='red', linestyle='--', label=f'Mean: {np.mean(dice_scores):.3f}')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

axes[1, 0].hist(iou_scores, bins=20, edgecolor='black', alpha=0.7, color='orange')
axes[1, 0].set_title('IoU Score Distribution')
axes[1, 0].set_xlabel('IoU Score')
axes[1, 0].set_ylabel('Frequency')
axes[1, 0].axvline(np.mean(iou_scores), color='red', linestyle='--', label=f'Mean: {np.mean(iou_scores):.3f}')
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

metrics_text = f"""
Model Performance Metrics on Test Set:

Test Loss: {test_loss:.4f}
Test Dice Coefficient: {test_dice:.4f}
Test IoU Score: {test_iou:.4f}

Mean Dice Score: {np.mean(dice_scores):.4f}
Mean IoU Score: {np.mean(iou_scores):.4f}

Confusion Matrix:
TN: {cm[0, 0]} | FP: {cm[0, 1]}
FN: {cm[1, 0]} | TP: {cm[1, 1]}

True Positive Rate: {cm[1, 1] / (cm[1, 1] + cm[1, 0]):.4f}
False Positive Rate: {cm[0, 1] / (cm[0, 1] + cm[0, 0]):.4f}
"""

axes[1, 1].text(0.1, 0.5, metrics_text, fontsize=11, verticalalignment='center',
                family='monospace', bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))
axes[1, 1].axis('off')

plt.tight_layout()
plt.show()

print("Comprehensive evaluation completed!")

In [ ]:
model.save('brain_tumor_segmentation_model.h5')
print("Model saved as 'brain_tumor_segmentation_model.h5'")